# Keyword trend modeling

Predicts next-year change in a word's share of thesis titles within a subject,
for words already flagged as rising by the EDA notebook's method.

**Design choices:**
- Rising-word candidates are selected using only data up to a cutoff year
  (`SELECT_END`), never data from the test window, so the model can't benefit
  from knowing which words turned out to matter.
- Specificity and other features are computed cumulatively (using only years
  up to and including the current one), not from all-years totals.
- The panel is zero-filled: a year where a word doesn't appear counts as
  share = 0, not a skipped row.
- Target is **change in share** (`dshare = share(t+1) - share(t)`), not the
  raw share, so metrics measure real predictive skill instead of rewarding
  the model for just copying last year's number.
- Sample weights are next-year title-word volume, so noisy small-subject-years
  count less.
- Baselines: zero-change (persistence) and momentum (`pred = slope_3yr`),
  both need to be beaten for the model to be worth anything.

**Inputs:** `data/word_year_subject_v5_final.parquet`,
`data/subject_meta_v5_final.parquet` (from `01_build_parquets.ipynb`).


In [ ]:
# %% [imports]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score
from sklearn.utils import shuffle

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120


In [ ]:
# %% [parameters -- change here only]
PIPELINE_VERSION = "v5_final"   # must match the build_parquets run this reads from

START_YEAR     = 1950
SELECT_END     = 2012   # rising words chosen using data <= this year (no future info)
SPLIT_YEAR     = 2013   # train < SPLIT_YEAR, test >= SPLIT_YEAR
TREND_END      = 2023   # last year of usable data
MIN_YEAR_WORDS = 50     # min title-words in a subject-year
MIN_TOTAL      = 10     # min cumulative count for a candidate word
SPEC_MIN       = 1.5    # subject share / global share floor
RECENT         = 10     # window (years) for the selection slope
TOP_K          = 10     # rising words kept per subject

assert SELECT_END < SPLIT_YEAR, "selection must not see the test window"

FEATURES = ["share_lag1", "share_lag2", "share_lag3",
            "slope_3yr", "slope_5yr", "accel", "vol_5yr",
            "spec_cum", "word_age", "log_volume"]


In [ ]:
# %% [load]
raw  = pd.read_parquet(f"data/word_year_subject_{PIPELINE_VERSION}.parquet")
meta = pd.read_parquet(f"data/subject_meta_{PIPELINE_VERSION}.parquet")

df = raw[
    (raw["year"] >= START_YEAR) &
    (raw["year"] <= TREND_END) &
    (raw["total_title_words"] >= MIN_YEAR_WORDS)
].copy()

# one row per (subject, year): token totals (repeated across words in the parquet)
subj_year_tot = (
    df.groupby(["subject", "year"])["total_title_words"]
    .first().rename("total").reset_index()
)
print(f"Rows: {len(df):,} | subjects: {df['subject'].nunique()} | years {df['year'].min()}-{df['year'].max()}")


In [ ]:
# %% [leak-free rising-word selection]
# Same idea as the EDA notebook's rising-word detection, but everything here
# is computed strictly from data <= end_year, so selecting candidates never
# uses information the test window wouldn't have had yet.

def select_rising_pairs(d, syt, end_year,
                        recent=RECENT, min_total=MIN_TOTAL,
                        spec_min=SPEC_MIN, top_k=TOP_K):
    d = d[d["year"] <= end_year]

    # cumulative counts up to end_year
    pair_tot  = d.groupby(["subject", "word"])["count"].sum()
    subj_tot  = d.groupby("subject")["count"].sum()
    glob_word = d.groupby("word")["count"].sum()
    glob_tot  = glob_word.sum()

    cand = pair_tot[pair_tot >= min_total].reset_index()
    subj_share = cand["count"] / cand["subject"].map(subj_tot)
    glob_share = cand["word"].map(glob_word) / glob_tot
    cand["spec"] = subj_share / np.maximum(glob_share, 1e-9)
    cand = cand[cand["spec"] >= spec_min]

    # recent slope on zero-filled shares (vectorized least squares)
    years = np.arange(end_year - recent + 1, end_year + 1)
    dr = d[d["year"].isin(years)]
    counts = (
        dr.pivot_table(index=["subject", "word"], columns="year",
                       values="count", aggfunc="sum")
        .reindex(columns=years).fillna(0.0)
    )
    counts = counts.join(
        pd.DataFrame(index=cand.set_index(["subject", "word"]).index),
        how="inner")
    totals = (
        syt[syt["year"].isin(years)]
        .pivot(index="subject", columns="year", values="total")
        .reindex(columns=years)
    )
    tot_aligned = totals.reindex(counts.index.get_level_values("subject")).values
    shares = counts.values / np.where(np.isnan(tot_aligned) | (tot_aligned == 0),
                                      np.nan, tot_aligned)
    valid = (~np.isnan(shares)).sum(axis=1) >= 5

    x  = np.arange(recent, dtype=float)
    xc = x - x.mean()
    sh = np.nan_to_num(shares, nan=0.0)          # missing subject-years -> 0 share
    slope = (sh @ xc) / (xc @ xc) * 10 * 100     # %-points per decade

    out = counts.index.to_frame(index=False)
    out["recent_slope"] = slope
    out = out[valid]
    out = out.merge(cand[["subject", "word", "spec", "count"]]
                    .rename(columns={"count": "total"}),
                    on=["subject", "word"])
    out["enriched"] = (out["recent_slope"] * out["spec"]
                       / np.log1p(glob_word[out["word"]].values))

    # underscore-aware prefix dedup within subject (keep higher total).
    # only merges words that share the same underscore-delimited token
    # prefix (e.g. learn/learning), so a unigram like "machine" can't
    # swallow an unrelated bigram like "machine_learning".
    def dedup(g):
        keep = []
        for w in g.sort_values("total", ascending=False)["word"]:
            w_parts = w.split("_")
            dominated = any(
                k.split("_") == w_parts[:len(k.split("_"))] or
                w_parts == k.split("_")[:len(w_parts)]
                for k in keep
            )
            if not dominated:
                keep.append(w)
        return g[g["word"].isin(keep)]

    out = out.query("recent_slope > 0")
    out = pd.concat([dedup(g) for _, g in out.groupby("subject")],
                    ignore_index=True)
    return (out.sort_values("enriched", ascending=False)
            .groupby("subject").head(top_k)
            .reset_index(drop=True))

rising = select_rising_pairs(df, subj_year_tot, end_year=SELECT_END)
pairs  = rising[["subject", "word"]].drop_duplicates()
print(f"Rising pairs (selected as of {SELECT_END}): {len(pairs)} "
      f"across {pairs['subject'].nunique()} subjects")


In [ ]:
# %% [feature panel builder -- shared by training and the watchlist]

def rolling_slope(s, window):
    def _s(x):
        if np.any(np.isnan(x)):
            return np.nan
        return np.polyfit(range(len(x)), x, 1)[0]
    return s.rolling(window).apply(_s, raw=True)

def build_feature_panel(d, syt, pair_df):
    """Zero-filled (subject, word, year) panel with leak-free features.
    All features at year t use information from years <= t only."""
    grid = pair_df.merge(syt, on="subject")
    pn = (grid.merge(d[["subject", "word", "year", "count"]],
                     on=["subject", "word", "year"], how="left")
          .fillna({"count": 0}))
    pn["share"] = pn["count"] / pn["total"]
    pn = pn.sort_values(["subject", "word", "year"]).reset_index(drop=True)

    g = pn.groupby(["subject", "word"], sort=False)
    pn["share_lag1"] = g["share"].shift(1)
    pn["share_lag2"] = g["share"].shift(2)
    pn["share_lag3"] = g["share"].shift(3)
    pn["slope_3yr"]  = g["share"].transform(lambda s: rolling_slope(s, 3))
    pn["slope_5yr"]  = g["share"].transform(lambda s: rolling_slope(s, 5))
    pn["accel"]      = pn["slope_3yr"] - pn["slope_5yr"]
    pn["vol_5yr"]    = g["share"].transform(lambda s: s.rolling(5).std())

    first_seen = (pn[pn["count"] > 0].groupby(["subject", "word"])["year"]
                  .min().rename("first_year"))
    pn = pn.merge(first_seen, on=["subject", "word"], how="left")
    pn["word_age"]   = (pn["year"] - pn["first_year"]).clip(lower=0)
    pn["log_volume"] = np.log1p(pn["total"])

    # cumulative (leak-free) specificity
    g = pn.groupby(["subject", "word"], sort=False)
    pn["cum_count"] = g["count"].cumsum()
    sc = syt.sort_values(["subject", "year"]).copy()
    sc["subj_cum_tot"] = sc.groupby("subject")["total"].cumsum()
    pn = pn.merge(sc[["subject", "year", "subj_cum_tot"]], on=["subject", "year"])

    words = pair_df["word"].unique()
    gw = (d[d["word"].isin(words)]
          .groupby(["word", "year"])["count"].sum().rename("gcount").reset_index())
    yg = (pn[["word", "year"]].drop_duplicates()
          .merge(gw, on=["word", "year"], how="left").fillna({"gcount": 0})
          .sort_values(["word", "year"]))
    yg["g_cum"] = yg.groupby("word")["gcount"].cumsum()
    gt = d.groupby("year")["count"].sum().rename("gtot").reset_index().sort_values("year")
    gt["gtot_cum"] = gt["gtot"].cumsum()
    yg = yg.merge(gt[["year", "gtot_cum"]], on="year")
    pn = pn.merge(yg[["word", "year", "g_cum", "gtot_cum"]], on=["word", "year"])
    pn["spec_cum"] = (
        (pn["cum_count"] / pn["subj_cum_tot"].clip(lower=1))
        / np.maximum(pn["g_cum"] / pn["gtot_cum"].clip(lower=1), 1e-9)
    )

    # target and weight
    g = pn.groupby(["subject", "word"], sort=False)
    pn["share_next"] = g["share"].shift(-1)
    pn["total_next"] = g["total"].shift(-1)
    pn["dshare"]     = pn["share_next"] - pn["share"]
    return pn

panel = build_feature_panel(df, subj_year_tot, pairs)
dm = panel.dropna(subset=FEATURES + ["dshare", "total_next"]).copy()
print(f"Panel rows: {len(panel):,} | model-ready rows: {len(dm):,}")


In [ ]:
# %% [train/test split]
train = dm[dm["year"] <  SPLIT_YEAR]
test  = dm[dm["year"] >= SPLIT_YEAR]
print(f"Train: {len(train):,} rows | {train['year'].min()}-{train['year'].max()}")
print(f"Test:  {len(test):,} rows | {test['year'].min()}-{test['year'].max()}")

X_tr, y_tr, w_tr = train[FEATURES].values, train["dshare"].values, train["total_next"].values
X_te, y_te       = test[FEATURES].values,  test["dshare"].values

def report(name, pred):
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae  = mean_absolute_error(y_te, pred)
    nz   = y_te != 0
    dir_acc = (np.sign(pred[nz]) == np.sign(y_te[nz])).mean()
    print(f"{name:<22} RMSE={rmse:.6f}  MAE={mae:.6f}  dir_acc={dir_acc:.1%}")
    return rmse, mae, dir_acc


In [ ]:
# %% [baselines]
_ = report("zero-change (lag-1)", np.zeros(len(y_te)))
_ = report("momentum (slope_3yr)", test["slope_3yr"].values)


In [ ]:
# %% [ridge, volume-weighted]
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

ridge = Ridge(alpha=1.0)
ridge.fit(X_tr_s, y_tr, sample_weight=w_tr)
pred_ridge = ridge.predict(X_te_s)
_ = report("Ridge (weighted)", pred_ridge)

print(pd.DataFrame({"feature": FEATURES, "coef": ridge.coef_})
      .sort_values("coef", key=abs, ascending=False).to_string(index=False))


In [ ]:
# %% [gradient boosting]
gbr = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
                                    max_iter=400, random_state=42)
gbr.fit(X_tr, y_tr, sample_weight=w_tr)
pred_gbr = gbr.predict(X_te)
_ = report("HistGBR (weighted)", pred_gbr)


In [ ]:
# %% [leakage checks]
# 1. shuffled target (feature-target leakage)
r_sh = Ridge(alpha=1.0)
r_sh.fit(X_tr_s, shuffle(y_tr, random_state=42), sample_weight=w_tr)
rmse_sh = np.sqrt(mean_squared_error(y_te, r_sh.predict(X_te_s)))
print(f"Shuffled-target RMSE = {rmse_sh:.6f} (should be >= zero-change RMSE)")

# 2. selection-leak probe: what happens if candidates are picked using the
#    FULL window (including the test years) instead of SELECT_END only
rising_leaky = select_rising_pairs(df, subj_year_tot, end_year=TREND_END)
overlap = len(pairs.merge(rising_leaky[["subject", "word"]], on=["subject", "word"]))
print(f"Overlap clean vs full-window selection: {overlap}/{len(pairs)} pairs "
      f"({overlap/len(pairs):.0%}) -- the gap is what selecting with future "
      f"data would have quietly taken advantage of")


In [ ]:
# %% [ranking evaluation on CHANGE, per subject-year]
tc = test.copy()
tc["pred"] = pred_ridge

rows = []
for (subj, yr), grp in tc.groupby(["subject", "year"]):
    if len(grp) < 5:
        continue
    rel = grp["dshare"].values
    rel = rel - rel.min()                      # ndcg needs non-negative relevance
    rows.append({"subject": subj, "year": yr,
                 "ndcg_model": ndcg_score([rel], [grp["pred"].values]),
                 "ndcg_momentum": ndcg_score([rel], [grp["slope_3yr"].values]),
                 "n": len(grp)})
nd = pd.DataFrame(rows)
print(f"NDCG on change  model:    {nd['ndcg_model'].mean():.3f}")
print(f"NDCG on change  momentum: {nd['ndcg_momentum'].mean():.3f}")
print(f"(groups: {len(nd)})")


In [ ]:
# %% [QA + residual diagnostics]
assert train["year"].max() < SPLIT_YEAR and test["year"].min() >= SPLIT_YEAR
assert SELECT_END < SPLIT_YEAR
assert dm.duplicated(subset=["subject", "word", "year"]).sum() == 0
assert dm["share"].between(0, 1).all() and dm["share_next"].between(0, 1).all()
print("QA pass: split clean, selection precedes test, no dupes, shares in [0,1]")

tc["residual"] = y_te - pred_ridge
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(pred_ridge, tc["residual"], alpha=0.3, s=8)
axes[0].axhline(0, color="red", lw=1)
axes[0].set_xlabel("Predicted dshare"); axes[0].set_title("Residual vs predicted")
tc.groupby("year")["residual"].mean().plot(ax=axes[1], marker="o")
axes[1].axhline(0, color="red", lw=1); axes[1].set_title("Mean residual by year")
axes[2].hist(tc["residual"], bins=50, color="steelblue")
axes[2].set_title("Residual distribution")
sns.despine(); plt.tight_layout()
plt.savefig("images/model_residual_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# %% [multi-window experiment -- selection redone inside each window]
# Repeats the same leak-free pipeline at several different (select, split, end)
# cutoffs, to check the result isn't a fluke of one particular year choice.

def run_window(select_end, split_year, trend_end):
    d = raw[(raw["year"] >= START_YEAR) & (raw["year"] <= trend_end) &
            (raw["total_title_words"] >= MIN_YEAR_WORDS)].copy()
    syt = (d.groupby(["subject", "year"])["total_title_words"]
           .first().rename("total").reset_index())
    p  = select_rising_pairs(d, syt, end_year=select_end)[["subject", "word"]]
    pn = build_feature_panel(d, syt, p)
    dmw = pn.dropna(subset=FEATURES + ["dshare", "total_next"])

    tr, te = dmw[dmw["year"] < split_year], dmw[dmw["year"] >= split_year]
    if len(tr) < 200 or len(te) < 200:
        print(f"{select_end}/{split_year}/{trend_end}: too little data")
        return
    sc = StandardScaler()
    r  = Ridge(alpha=1.0)
    r.fit(sc.fit_transform(tr[FEATURES]), tr["dshare"], sample_weight=tr["total_next"])
    pr = r.predict(sc.transform(te[FEATURES]))
    rmse0 = np.sqrt(mean_squared_error(te["dshare"], np.zeros(len(te))))
    rmse  = np.sqrt(mean_squared_error(te["dshare"], pr))
    nz = (te["dshare"] != 0).values
    da = (np.sign(pr[nz]) == np.sign(te["dshare"].values[nz])).mean()
    print(f"sel<={select_end} split={split_year} end={trend_end} | "
          f"zero-RMSE={rmse0:.6f} ridge-RMSE={rmse:.6f} dir={da:.1%}")

for se, sp, en in [(2004, 2005, 2015), (2007, 2008, 2018),
                   (2009, 2010, 2020), (2012, 2013, 2023)]:
    run_window(se, sp, en)


In [ ]:
# %% [watchlist for TREND_END + 1 -- full data allowed here (deployment)]
# This is the only place full data (up to TREND_END) is used for selection,
# by design: we're making live forward predictions here, not evaluating.
rising_now = select_rising_pairs(df, subj_year_tot, end_year=TREND_END)
pairs_now  = rising_now[["subject", "word"]].drop_duplicates()

panel_now = build_feature_panel(df, subj_year_tot, pairs_now)
latest = panel_now[panel_now["year"] == TREND_END].dropna(subset=FEATURES).copy()

# refit on ALL leak-free panel rows for deployment
sc_f = StandardScaler()
r_f  = Ridge(alpha=1.0)
r_f.fit(sc_f.fit_transform(dm[FEATURES]), dm["dshare"], sample_weight=dm["total_next"])

latest["pred_dshare"] = r_f.predict(sc_f.transform(latest[FEATURES]))
latest["pred_share"]  = latest["share"] + latest["pred_dshare"]

watch = latest.merge(meta, on="subject")
top5 = (watch.sort_values("pred_dshare", ascending=False)
        .groupby("subject_name").head(5)
        [["subject_name", "word", "share", "pred_share", "pred_dshare",
          "slope_3yr", "spec_cum"]]
        .sort_values(["subject_name", "pred_dshare"], ascending=[True, False])
        .reset_index(drop=True))
top5.to_csv(f"data/watchlist_{TREND_END + 1}_{PIPELINE_VERSION}.csv", index=False)
print(f"Saved watchlist_{TREND_END + 1}_{PIPELINE_VERSION}.csv | {len(top5)} words, "
      f"{top5['subject_name'].nunique()} subjects")
top5.head(100)
